In [1]:
from BCP303.BCP303 import BPC303
from Sourcemeter.sourcemeter import Sourcemeter2401
import time
from datetime import datetime
from tool.tools import post_process


In [ ]:
# move the stage in Z direction
bcp303_z = BPC303(channel_id=1)
position_z = bcp303_z.move_to_origin(start_position=0)
step_size_z = 0.1  # move 0.1 mm
position_z = bcp303_z.bcp303_move_stage(step_size=step_size_z, current_position=position_z,)
bcp303_z.bcp303_stop(ifback=True)

In [12]:
# move the stage in x direction
bcp303_x = BPC303(channel_id=2)
position_x = bcp303_x.move_to_origin(start_position=0)
step_size_x = 0.1  # move 0.1 mm
position_x = bcp303_x.bcp303_move_stage(step_size=step_size_x, current_position=position_x,)
bcp303_x.bcp303_stop(ifback=False)

APT High Voltage Amplifier initialized
Stage Moving Done
Stage disconnected


In [ ]:
# record the signals
sm2401 = Sourcemeter2401(speed_nplc=0.1)


In [3]:
# sweep operation: move stage in z-direction and record the signals

def sweep_operation(stage_settings=None,chip_name="chip_test",sample_name="beam_test"):
    step_number = stage_settings["step_number"]
    time_interval = stage_settings["time_interval"]
    position_x = stage_settings["position_x"]
    step_size_z = stage_settings["step_size_z"]
    count = 1
    formatted_time = datetime.now().strftime("%Y%m%d%H%M")
    try:
        bcp303_z = BPC303(channel_id=1)
        bcp303_device = bcp303_z.get_device()
        # moving controller x
        bcp303_x = BPC303(channel_id=2, device=bcp303_device)
        # Sourcemeter
        sm2401 = Sourcemeter2401(speed_nplc=0.1)
        # set the initial position of the stage z
        position_z = bcp303_z.move_to_origin(start_position=stage_settings["position_z"]) - step_size_z
        time.sleep(1)
        bcp303_x.move_to_origin()
        time.sleep(1)
        allData = [{"position": [], "voltage": []}]
        for _ in range(step_number):
            position_z = bcp303_z.bcp303_move_stage(
                step_size=step_size_z,
                current_position=position_z,
            )
            allData[0]["position"].append(position_z)
            time.sleep(0.5)
            bcp303_x.move_to_position(start_position=0, step_size=0.5, final_position=position_x)
            time.sleep(0.5)
            step_start = time.perf_counter()
            voltage = sm2401.measure_voltage(duration=time_interval / 2, dt=0.01)[
                "voltage"
            ]
            allData[0]["voltage"].append(voltage)
            # remaining time to wait until the next step
            elapsed = time.perf_counter() - step_start
            remaining = time_interval - elapsed
            if remaining > 0:
                time.sleep(remaining)
            bcp303_x.move_to_origin()
            time.sleep(0.5)
            print(f"process completed: {100 * count / step_number:.1f}%")
            count += 1
        sm2401_settings = sm2401.getSettings()
        stage_settings["position_x"] = position_x
        settings = {"stage": stage_settings, "sourcemeter": sm2401_settings}
        post_process(
            chip_name=chip_name,
            sample_name=sample_name,
            result=allData[0],
            config=settings,
            repeat=1,
            position_z=position_z,
            ifshow=False,
            formatted_time=formatted_time,
        )
    # stop the stage and sourcemeter
    except Exception as e:
        print(f"Exception occurred: {e}")
    finally:
        time.sleep(0.5)
        sm2401.close()
        bcp303_z.move_to_origin()
        bcp303_z.channel.StopPolling()
        bcp303_x.bcp303_stop(ifback=True)
        return allData[0], settings
    

setting = {
        "position_x": 2,
        "step_number": 200,
        "position_z": 0,
        "step_size_z": 0.1,
        "time_interval": 2,  
    }
sweep_operation(
        stage_settings=setting,
        chip_name="V1_R_W_1_Left",
        # chip_name="SiN_beam",
        # sample_name="w2_sweep",  # test_1_right, w=20
        sample_name="test_AFM4_450_w20_4_sweep",  
    )

APT High Voltage Amplifier initialized
APT High Voltage Amplifier initialized
IDN: KEITHLEY INSTRUMENTS INC.,MODEL 2401,4614233,B02 Jan 20 2021 10:19:49/B01  /W/N
process completed: 0.5%
process completed: 1.0%
process completed: 1.5%
process completed: 2.0%
process completed: 2.5%
process completed: 3.0%
process completed: 3.5%
process completed: 4.0%
process completed: 4.5%
process completed: 5.0%
process completed: 5.5%
process completed: 6.0%
process completed: 6.5%
process completed: 7.0%
process completed: 7.5%
process completed: 8.0%
process completed: 8.5%
process completed: 9.0%
process completed: 9.5%
process completed: 10.0%
process completed: 10.5%
process completed: 11.0%
process completed: 11.5%
process completed: 12.0%
process completed: 12.5%
process completed: 13.0%
process completed: 13.5%
process completed: 14.0%
process completed: 14.5%
process completed: 15.0%
process completed: 15.5%
process completed: 16.0%
process completed: 16.5%
process completed: 17.0%
proces

({'position': [-0.000610370189519944,
   0.098269600512711,
   0.194097720267342,
   0.290536210211493,
   0.388195440534684,
   0.486465041047395,
   0.585955381939146,
   0.684835352641377,
   0.784325693533128,
   0.882595294045839,
   0.98147526474807,
   1.0803552354503,
   1.17862483596301,
   1.27811517685476,
   1.37760551774651,
   1.47587511825922,
   1.57414471877194,
   1.67180394909513,
   1.76946317941832,
   1.86712240974151,
   1.96417126987518,
   2.06427198095645,
   2.16254158146916,
   2.26203192236091,
   2.36152226325266,
   2.46040223395489,
   2.56050294503616,
   2.65755180516984,
   2.75460066530351,
   2.85042878505814,
   2.94503616443373,
   3.04452650532548,
   3.14218573564867,
   3.23923459578234,
   3.33750419629505,
   3.43577379680776,
   3.53526413769951,
   3.63414410840175,
   3.7336344492935,
   3.83373516037477,
   3.93444624164556,
   4.03454695272683,
   4.1346476638081,
   4.23474837488937,
   4.33484908597064,
   4.43433942686239,
   4.533829